In [ ]:
# you cannot monitor only the status of individual parking spaces to isolate 
# portions of a full video clip that contain vehicle maneuvers
# because a vehicle driving through the driving lane in front of the parking space will obscure all 
# parked vehicles behind it, disrupting tracking 
# and making it uncertain whether the status of behind cars changed

# track id will either not be assigned to many cars in the parking lot
# or will change when occluded and return to view

In [20]:
import cv2
import csv
import json

video_ID = '1FULL'
vid = cv2.VideoCapture(f'clips/Parking-clip{video_ID}.mp4')

motionThreshold = 50*50

with open('mappings.json', 'r') as f:
  mapping_data = json.load(f)
  region_ID = mapping_data.get(f'{video_ID}')

with open(f'regions{region_ID}.json', 'r') as f:
  region_data = json.load(f)
  parking_regions = region_data.get("parking_areas") 
  min_y = float('inf')
  max_y = 0 
  for region in parking_regions:
    min_y = min(min_y, min(point[1] for point in region))
    max_y = max(max_y, max(point[1] for point in region))

output_file = open(f'timestamps/timestamps{video_ID}.csv', 'w', newline='') 
writer = csv.DictWriter(output_file, fieldnames=["start_time", "end_time"])
writer.writeheader()

backsub = cv2.createBackgroundSubtractorMOG2(history=20*30, varThreshold=16, detectShadows=True)

start_frame = None
end_frame = None
frame_idx = 0
while True:
  ret, frame = vid.read()
  if not ret:
    break

  frame_idx+=1
  cropped_frame = frame[min_y:max_y, :]

  foreground = backsub.apply(cropped_frame)
  contours, __ = cv2.findContours(foreground, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

  motion_detected = False
  for contour in contours: 
    if cv2.contourArea(contour) >= motionThreshold:
      motion_detected = True
      break

  if motion_detected:
    if start_frame is None:
      start_frame = frame_idx
  else: 
    if start_frame is not None:
      end_frame = frame_idx
      writer.writerow({
        "start_time": start_frame,
        "end_time": end_frame + 30*5
      })
      start_frame = None
      end_frame = None

output_file.close()


In [18]:
import pandas as pd
import cv2
import time

video_ID = '1FULL'
key_events_record = f'timestamps/timestamps{video_ID}.csv'
vid = cv2.VideoCapture(f'clips/Parking-clip{video_ID}.mp4')

key_events = pd.read_csv(key_events_record)
for index, row in key_events.iterrows():
  vid.set(cv2.CAP_PROP_POS_FRAMES, int(row["start_time"]))
  for frame_idx in range(int(row["start_time"]), int(row["end_time"])):
    ret, frame = vid.read()
    cropped_frame = frame[frame.shape[0]//3:, :]
    cv2.imshow("Highlights", cropped_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        vid.release()
        cv2.destroyAllWindows()
        break
    time.sleep(0.01)

vid.release()
cv2.destroyAllWindows()
    

In [ ]:
import cv2
import torch
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from boxmot import BotSort
import csv
import json
import pandas as pd

video_ID = '1FULL'
movement_threshold = 50
time_resolution = 0.1

FPS = 30
SKIP = time_resolution*FPS

vid = cv2.VideoCapture(f'clips/Parking-clip{video_ID}')
device = torch.device('cpu')
model = YOLO("/home/edisonz/maneuver_heuristics25/yolov8x.pt")
tracker = BotSort(reid_weights=Path('clip_vehicleid.pt'), device=device, half=False)

key_timestamps = pd.read_csv(f'timestamps/timestamps{video_ID}')

with open("mappings.json", "r") as f:
  mapping_data = json.load(f)   
  LR_ID = mapping_data.get(f'{video_ID}')

with open(f"regions{LR_ID}", "r") as f:
  regions_data = json.load(f)

parking_areas = [
    np.array(poly, dtype=np.int32).reshape((-1, 1, 2))
    for poly in regions_data.get("parking_areas", [])
]

def in_parking_area(point):
    return any(cv2.pointPolygonTest(polygon, point, False) >= 0 for polygon in parking_areas)

for index, row in key_timestamps.iterrows():
  vid.set(cv2.CAP_PROP_POS_FRAMES, row["start_time"])
  vehicle_tracks = {}
  saved_tracks = []
  for i in range(row["start_time"], row["end_time"]):
    if (i-row["start_time"])%SKIP!=0:
      continue

    ret, frame = vid.read()
    cropped_frame = frame[frame.shape[0]//3:, :]

    results = model(cropped_frame)[0]

    dets = []
    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = box.conf[0].cpu().item()
        cls_id = int(box.cls[0].cpu().item())
        if cls_id != 2 and cls_id != 7:
            continue
        dets.append([x1, y1, x2, y2, conf, cls_id])

        dets = np.array(dets)
        tracks = tracker.update(dets, cropped_frame)

    for track in tracks:
      x1, y1, x2, y2, track_id = track[:5]
      if track_id not in vehicle_tracks: 
        vehicle_tracks[track_id] = [{
           "x": (x1 + x2)//2,
           "y": (y1 + y2)//2,
           "in_parking_zone": in_parking_area((x1 + x2)//2, y2),
           "height": y2 - y1,
           "width": x2 - x1
        }]
      else:
        vehicle_tracks[track_id].append({
          "x": (x1 + x2)//2,
          "y": (y1 + y2)//2,
          "in_parking_zone": in_parking_area((x1 + x2)//2, y2),
          "height": y2 - y1,
          "width": x2 - x1  
        })


In [ ]:
import cv2
import torch
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from boxmot import BotSort
import csv
import json
from enum import Enum

video_ID = '92ENT'

# Setup
input_size = [800, 1440]
device = torch.device('cpu')
model = YOLO("/home/edisonz/maneuver_heuristics25/yolov8x.pt")
tracker = BotSort(reid_weights=Path('clip_vehicleid.pt'), device=device, half=False)
vid = cv2.VideoCapture(f'/home/edisonz/maneuver_heuristics25/clips/Parking-clip{video_ID}.mp4')
FPS = 30

frame_count = 0
while True:
    ret, full_frame = vid.read()
    if not ret:
        break

    frame_count += 1
    if frame_count % SKIP != 0:
        continue

    height = full_frame.shape[0]
    cropped_frame = full_frame[height//3:, :]

    results = model(cropped_frame)[0]

    dets = []
    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = box.conf[0].cpu().item()
        cls_id = int(box.cls[0].cpu().item())
        if cls_id != 2 and cls_id != 7:
            continue
        dets.append([x1, y1, x2, y2, conf, cls_id])

    dets = np.array(dets)
    tracks = tracker.update(dets, cropped_frame)

    for track in tracks:
        x1, y1, x2, y2, track_id = track[:5]
        if(track_id!=selected_track):
            continue
        
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        bottom = (int(cx), int(y2))  # bottom-center

        in_parking = None
        if(maneuver_type==ManeuverType.ENT):
            if(in_parking_area((int(x1),int(y2))) or in_parking_area((int(x2),int(y2)))):
                in_parking = 1
            else: 
                in_parking = 0
        if(maneuver_type==ManeuverType.EXT):
            if(in_parking_area((int(x1),int(y2))) or in_parking_area((int(x2),int(y2)))):
                in_parking = 1
            else:
                in_parking = 0

        height = y2 - y1
        width = x2 - x1

        writer.writerow({
            "frame": frame_count,
            "track_id": int(track_id),
            "x": cx,
            "y": cy,
            "in_parking_zone": in_parking,
            "height": height,
            "width": width
        })

# Cleanup
vid.release()
csv_file.close()
cv2.destroyAllWindows() 

In [ ]:
import cv2
import pandas as pd
import time

video_ID = '1ENT'
vid = cv2.VideoCapture(f'clips/Parking-clip{video_ID}.mp4')

track_history = pd.read_csv(f'clips/Parking-clip{video_ID}.csv')

captured_frames = track_history['frame'].unique()

for frame_index in captured_frames:
  vid.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
  print(frame_index)

  ret, frame = vid.read()
  cropped_frame = frame[frame.shape[0]//3:, :] 

  current_frame = track_history[track_history['frame']==frame_index]

  for index, row in current_frame.iterrows():
    x1 = int(row['x'] - row['width']//2) 
    x2 = int(row['x'] + row['width']//2)
    y1 = int(row['y'] - row['height']//2)
    y2 = int(row['y'] + row['height']//2)
    tracking_box = [(x1, y1), (x2, y1), (x1, y2), (x2, y2)]

    color = None
    if row['in_parking_zone']==0:
      color = (0, 255, 0)
    if row['in_parking_zone']==1:
      color = (0, 0, 255)

    cv2.line(cropped_frame, (x1, y1), (x2, y1), color, 2)
    cv2.line(cropped_frame, (x2, y1), (x2, y2), color, 2)
    cv2.line(cropped_frame, (x2, y2), (x1, y2), color, 2)
    cv2.line(cropped_frame, (x1, y2), (x1, y1), color, 2)

    cv2.putText(cropped_frame, f"{row['track_id']}", (x1, y2 + 20), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    cv2.imshow('Replay', cropped_frame)
    time.sleep(0.1)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        vid.release()
        cv2.destroyAllWindows()
        break

vid.release()

#for row in len(track_history):
#  
#
#  vid.set(cv2.CAP_PROP_POS_FRAMES, )
#  ret, frame = vid.read()
#  if not ret:
#    break
#
#  cropped_frame = frame.copy()[frame.shape[0]//3:,:]
#
#  cv2.imshow("Tracking playback", cropped_frame)
#
#